In [1]:
import os
from pyspark.sql import SparkSession
import pandas as pd

# Set JAVA_HOME for PySpark
os.environ['JAVA_HOME'] = '/opt/homebrew/Cellar/openjdk@21/21.0.9/libexec/openjdk.jdk/Contents/Home'

# Stop any existing spark sessions
SparkSession._instantiatedSession = None if SparkSession._instantiatedSession else None

In [2]:
from pyspark.sql.functions import current_timestamp, regexp_replace, col, when, coalesce, lit, length

In [3]:
spark = SparkSession.builder \
    .appName("LendingClub") \
    .master("local[2]") \
    .enableHiveSupport() \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/01/29 13:12:50 WARN Utils: Your hostname, Himanshus-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.215 instead (on interface en0)
26/01/29 13:12:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/29 13:12:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Customers

In [ ]:
file_path = "../data/raw/customers.csv"

In [ ]:
customers_df = spark.read\
        .format("csv")\
        .option("header", "true")\
        .option("inferSchema", "true")\
        .load(file_path)

In [ ]:
customers_df.show(5)

In [ ]:
customers_df_renamed = customers_df.withColumnRenamed("annual_inc", "annual_income")\
                            .withColumnRenamed("addr_state", "address_state")\
                            .withColumnRenamed("zip_code", "address_zipcode")\
                            .withColumnRenamed("country", "address_country")\
                            .withColumnRenamed("tot_hi_cred_lim", "total_high_credit_limit")\
                            .withColumnRenamed("annual_inc_joint", "join_annual_income")

In [ ]:
customers_df_ingested = customers_df_renamed.withColumn("ingest_date", current_timestamp())

In [ ]:
customers_df_ingested.printSchema()

In [ ]:
customers_df_distinct = customers_df_ingested.distinct()

In [ ]:
customers_df_distinct.createOrReplaceTempView("customers")

In [ ]:
customers_income_filtered = spark.sql("select * from customers where annual_income is not null")

In [ ]:
customers_income_filtered.show(5)

In [ ]:
customers_income_filtered.createOrReplaceTempView("customers")

In [ ]:
spark.sql("select distinct(emp_length) from customers").show()

In [ ]:
spark.sql("SELECT regexp_replace(emp_length, '[^0-9]', '') AS emp_length FROM customers").show()

In [ ]:
customers_emplength_cleaned = customers_income_filtered.withColumn("emp_length", regexp_replace(col("emp_length"), "[^0-9]",""))

In [ ]:
customers_emplength_cleaned.show(5)

In [ ]:
customers_emplength_cleaned.printSchema()

In [ ]:
#Replace empty strings with NULL before casting
customers_emplength_int = customers_emplength_cleaned.withColumn(
    "emp_length",
    when(col("emp_length") == "", None).otherwise(col("emp_length")).cast('int')
)

In [ ]:
customers_emplength_int.createOrReplaceTempView("customers")

In [ ]:
avg_emp_length = spark.sql("select floor(avg(emp_length)) as avg_emp_length from customers").collect()
print(avg_emp_length)

In [ ]:
avg_emp_duration = avg_emp_length[0][0]
print(avg_emp_duration)

In [ ]:
customers_emplength_replaced = customers_emplength_int.na.fill(avg_emp_duration, subset=['emp_length'])

In [ ]:
customers_emplength_replaced.createOrReplaceTempView("customers")
spark.sql("select distinct(address_state) from customers").show()

In [ ]:
spark.sql("select count(address_state) from customers where length(address_state)>2").show()

In [ ]:
customers_state_cleaned = customers_emplength_replaced.withColumn(
    "address_state",
    when(length(col("address_state"))> 2, "NA").otherwise(col("address_state"))
)

In [ ]:
customers_state_cleaned.show()

In [ ]:
customers_state_cleaned.write \
.option("header", True) \
.format("csv") \
.mode("overwrite") \
.option("path", "../data/cleaned/customers.csv") \
.save()

Loans

In [ ]:
loans_schema = 'loan_id string, member_id string, loan_amount float, funded_amount float, loan_term_months string, interest_rate float, monthly_installment float, issue_date string, loan_status string, loan_purpose string, loan_title string'

In [ ]:
loans_raw_df = spark.read \
.format("csv") \
.option("header",True) \
.schema(loans_schema) \
.load("../data/raw/loans.csv")

loans_raw_df.show(5)

In [ ]:
loans_raw_df.printSchema()

In [ ]:
loans_df_ingestd = loans_raw_df.withColumn("ingest_date", current_timestamp())

In [ ]:
loans_df_ingestd.createOrReplaceTempView("loans")

In [ ]:
spark.sql("select * from loans where loan_amount is null").show(5)


In [ ]:
columns_with_na = ["loan_amount", "funded_amount", "loan_term_months", "interest_rate", "monthly_installment", "issue_date", "loan_status", "loan_purpose", "loan_title"]

In [ ]:
loans_filtered_df = loans_df_ingestd.na.drop(subset = columns_with_na)

In [ ]:
loans_filtered_df.createOrReplaceTempView("loans")

In [ ]:
loans_term_modified_df = loans_filtered_df.withColumn("loan_term_months", 
                                                      (regexp_replace(col("loan_term_months"), " months", "") \
.cast("int") / 12).cast("int")).withColumnRenamed("loan_term_months","loan_term_years")

In [ ]:
loans_term_modified_df.show(5)
loans_term_modified_df.printSchema()

In [ ]:
loans_term_modified_df.createOrReplaceTempView("loans")

In [ ]:
spark.sql("select distinct(loan_purpose) from loans").show()

In [ ]:
spark.sql("select loan_purpose, count(*) as total from loans group by loan_purpose order by total desc").show()

In [ ]:
loan_purposes_valid = ["debt_consolidation", "credit_card", "home_improvement", "other", "major_purchase", "medical", "small_business", "car", "vacation", "moving", "house", "wedding", "renewable_energy", "educational"]

In [ ]:
loans_purpose_modified = loans_term_modified_df.withColumn("loan_purpose", when(col("loan_purpose").isin(loan_purposes_valid), col("loan_purpose")).otherwise("other"))

In [ ]:
loans_purpose_modified.createOrReplaceTempView("loans")

In [ ]:
spark.sql("select loan_purpose, count(*) as total from loans group by loan_purpose order by total desc").show()

In [ ]:
loans_purpose_modified.write \
.option("header", True) \
.format("csv") \
.mode("overwrite") \
.option("path", "../data/cleaned/loans.csv")\
.save()

Loan repayments

In [ ]:
loans_repay_raw_df = spark.read \
.format("csv") \
.option("header",True) \
.option("inferSchema", True) \
.load("../data/raw/repayments.csv")

loans_repay_raw_df.show()

In [ ]:
loans_repay_raw_df.printSchema()

In [ ]:
loans_repay_schema = 'loan_id string, total_principal_received float, total_interest_received float, total_late_fee_received float, total_payment_received float, last_payment_amount float, last_payment_date string, next_payment_date string'

In [ ]:
loans_repay_raw_df = spark.read \
.format("csv") \
.option("header",True) \
.schema(loans_repay_schema) \
.load("../data/raw/repayments.csv")

loans_repay_raw_df.printSchema()

In [ ]:
loans_repay_df_ingestd = loans_repay_raw_df.withColumn("ingest_date", current_timestamp())

loans_repay_df_ingestd.printSchema()

In [ ]:
loans_repay_df_ingestd.createOrReplaceTempView("loan_repayments")

In [ ]:
spark.sql("select count(*) from loan_repayments where total_principal_received is null").show()

In [ ]:
columns_to_check = ["total_principal_received", "total_interest_received", "total_late_fee_received", "total_payment_received"]

In [ ]:
loans_repay_filtered_df = loans_repay_df_ingestd.na.drop(subset=columns_to_check)

In [ ]:
loans_repay_filtered_df.createOrReplaceTempView("loan_repayments")

In [ ]:
spark.sql("select count(*) from loan_repayments where total_payment_received = 0.0").show()

In [ ]:
spark.sql("select count(*) from loan_repayments where total_payment_received = 0.0 and total_principal_received != 0.0").show()

In [ ]:
loans_payments_fixed_df = loans_repay_filtered_df.withColumn(
   "total_payment_received",
    when(
        (col("total_principal_received") != 0.0) &
        (col("total_payment_received") == 0.0),
        col("total_principal_received") + col("total_interest_received") + col("total_late_fee_received")
    ).otherwise(col("total_payment_received"))
)

In [ ]:
loans_payments_fixed_df.show()

In [ ]:
loans_payments_fixed2_df = loans_payments_fixed_df.filter("total_payment_received != 0.0")

In [ ]:
loans_payments_fixed2_df.createOrReplaceTempView("loan_repayments")

In [ ]:
spark.sql("select last_payment_date, count(*) as total from loan_repayments group by last_payment_date order by total").show(178)

In [ ]:
loans_payments_ldate_fixed_df = loans_payments_fixed2_df.withColumn(
    "next_payment_date",
    when(col("last_payment_date").rlike(r'^\d+(\.\d+)?$'), None)\
        .otherwise(col("last_payment_date"))
)

In [ ]:
loans_payments_ndate_fixed_df = loans_payments_ldate_fixed_df.withColumn(
  "last_payment_date",
   when(
       (col("next_payment_date").rlike(r'^\d+(\.\d+)?$')),
       None
       ).otherwise(col("next_payment_date"))
)

In [ ]:
loans_payments_ndate_fixed_df.write \
.option("header", True) \
.format("csv") \
.mode("overwrite") \
.option("path", "../data/cleaned/repayments.csv")\
.save()

Cleaning delinquencies

In [ ]:
loans_def_raw_df = spark.read \
.format("csv") \
.option("header",True) \
.option("inferSchema", True) \
.load("../data/raw/delinquencies.csv")

In [ ]:
loans_def_raw_df.show()

In [ ]:
loans_def_raw_df.printSchema()

In [ ]:
loans_def_raw_df.createOrReplaceTempView("loan_defaulters")

In [ ]:
spark.sql("select delinq_2yrs, count(*) as total from loan_defaulters group by delinq_2yrs order by total desc").show()

In [ ]:
loan_defaulters_schema = "member_id string, delinq_2yrs float, delinq_amnt float, pub_rec float, pub_rec_bankruptcies float,inq_last_6mths float, total_rec_late_fee float, mths_since_last_delinq float, mths_since_last_record float"

In [ ]:
loans_def_raw_df = spark.read \
.format("csv") \
.option("header",True) \
.schema(loan_defaulters_schema) \
.load("../data/raw/delinquencies.csv")

In [ ]:
loans_def_raw_df.createOrReplaceTempView("loan_defaulters")

In [ ]:
loans_def_processed_df = loans_def_raw_df.withColumn("delinq_2yrs", col("delinq_2yrs").cast("integer")).fillna(0, subset = ["delinq_2yrs"])

In [ ]:
loans_def_processed_df.createOrReplaceTempView("loan_defaulters")

In [ ]:
spark.sql("select count(*) from loan_defaulters where delinq_2yrs is null").show()

In [ ]:
loans_def_delinq_df = spark.sql("select member_id,delinq_2yrs, delinq_amnt, int(mths_since_last_delinq) from loan_defaulters where delinq_2yrs > 0 or mths_since_last_delinq > 0")

In [ ]:
loans_def_delinq_df.show()

In [ ]:
loans_def_records_enq_df = spark.sql("select member_id from loan_defaulters where pub_rec > 0.0 or pub_rec_bankruptcies > 0.0 or inq_last_6mths > 0.0")

In [ ]:
loans_def_records_enq_df.show()

In [ ]:
loans_def_delinq_df.write \
.option("header", True) \
.format("csv") \
.mode("overwrite") \
.option("path", "../data/cleaned/delinquencies.csv") \
.save()

In [ ]:
loans_def_records_enq_df.write \
.option("header", True) \
.format("csv") \
.mode("overwrite") \
.option("path", "../data/cleaned/delinquencies_public_records.csv") \
.save()

Creating Defaulters details

In [ ]:
loans_def_p_pub_rec_df = loans_def_processed_df.withColumn("pub_rec", col("pub_rec").cast("integer")).fillna(0, subset = ["pub_rec"])

In [ ]:
loans_def_p_pub_rec_bankruptcies_df = loans_def_p_pub_rec_df.withColumn("pub_rec_bankruptcies", col("pub_rec_bankruptcies").cast("integer")).fillna(0, subset = ["pub_rec_bankruptcies"])

In [ ]:
loans_def_p_inq_last_6mths_df = loans_def_p_pub_rec_bankruptcies_df.withColumn("inq_last_6mths", col("inq_last_6mths").cast("integer")).fillna(0, subset = ["inq_last_6mths"])

In [ ]:
loans_def_p_inq_last_6mths_df.createOrReplaceTempView("loan_defaulters")

In [ ]:
loans_def_detail_records_enq_df = spark.sql("select member_id, pub_rec, pub_rec_bankruptcies, inq_last_6mths from loan_defaulters")

In [ ]:
loans_def_detail_records_enq_df.show()

In [ ]:
loans_def_detail_records_enq_df.write \
.option("header", True) \
.format("csv") \
.mode("overwrite") \
.option("path", "../data/cleaned/loans_del_detail_records_enq_df.csv") \
.save()

Create permanent tables on the datasets

In [4]:
spark.sql("create database lending_club")

26/01/29 13:13:24 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
26/01/29 13:13:24 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore himanshuhegde@127.0.0.1


AnalysisException: [SCHEMA_ALREADY_EXISTS] Cannot create schema `lending_club` because it already exists.
Choose a different name, drop the existing schema, or add the IF NOT EXISTS clause to tolerate pre-existing schema. SQLSTATE: 42P06

In [ ]:
spark.sql("drop table if exists lending_club.loans")

In [ ]:
spark.sql("""create external table lending_club.customers(
member_id string, emp_title string, emp_length int, 
home_ownership string, annual_income float, address_state string, address_zipcode string, address_country string, grade string, 
sub_grade string, verification_status string, total_high_credit_limit float, application_type string, join_annual_income float, 
verification_status_joint string, ingest_date timestamp)
using csv location  '/Users/himanshuhegde/Desktop/Lending_club/data/cleaned/customers.csv'
options('header'='true', 'inferSchema'='false')
""")

In [ ]:
spark.sql("select * from lending_club.customers").show()

In [ ]:
spark.sql("""
create external table lending_club.loans(
loan_id string, member_id string, loan_amount float, funded_amount float,
loan_term_years integer, interest_rate float, monthly_installment float, issue_date string,
loan_status string, loan_purpose string, loan_title string, ingest_date timestamp)
using csv location  '/Users/himanshuhegde/Desktop/Lending_club/data/cleaned/loans.csv'
options('header'='true', 'inferSchema'='false')
""")

In [ ]:
spark.sql("select * from lending_club.loans").show()

In [ ]:
spark.sql("""CREATE EXTERNAL TABLE lending_club.loans_repayments(
    loan_id string, total_principal_received float, 
    total_interest_received float,total_late_fee_received float,
    total_payment_received float,last_payment_amount float,
    last_payment_date string,next_payment_date string,
    ingest_date timestamp)
    using csv location  '/Users/himanshuhegde/Desktop/Lending_club/data/cleaned/repayments.csv'
    options('header'='true', 'inferSchema'='false')
""")

In [ ]:
spark.sql("select * from lending_club.loans_repayments").show()

In [ ]:
spark.sql("""CREATE EXTERNAL TABLE lending_club.loans_defaulters_delinq(
    member_id string, delinq_2yrs integer, delinq_amnt float,
    mths_since_last_delinq integer)
    using csv location  '/Users/himanshuhegde/Desktop/Lending_club/data/cleaned/delinquencies.csv'
    options('header'='true', 'inferSchema'='false')
""")

In [ ]:
spark.sql("select * from lending_club.loans_defaulters_delinq").show()

In [ ]:
spark.sql("""CREATE EXTERNAL TABLE lending_club.loans_defaulters_detail_rec_enq(
    member_id string, pub_rec integer, pub_rec_bankruptcies integer, 
    inq_last_6mths integer)
    using csv location  '/Users/himanshuhegde/Desktop/Lending_club/data/cleaned/loans_del_detail_records_enq_df.csv'
    options('header'='true', 'inferSchema'='false')
    """)

In [ ]:
spark.sql("select * from lending_club.loans_defaulters_detail_rec_enq").show()

In [ ]:
spark.sql("""
create or replace view lending_club.customers_loan_v as select
l.loan_id,
c.member_id,
c.emp_title,
c.emp_length,
c.home_ownership,
c.annual_income,
c.address_state,
c.address_zipcode,
c.address_country,
c.grade,
c.sub_grade,
c.verification_status,
c.total_high_credit_limit,
c.application_type,
c.join_annual_income,
c.verification_status_joint,
l.loan_amount,
l.funded_amount,
l.loan_term_years,
l.interest_rate,
l.monthly_installment,
l.issue_date,
l.loan_status,
l.loan_purpose,
r.total_principal_received,
r.total_interest_received,
r.total_late_fee_received,
r.last_payment_date,
r.next_payment_date,
d.delinq_2yrs,
d.delinq_amnt,
d.mths_since_last_delinq,
e.pub_rec,
e.pub_rec_bankruptcies,
e.inq_last_6mths

FROM lending_club.customers c
LEFT JOIN lending_club.loans l on c.member_id = l.member_id
LEFT JOIN lending_club.loans_repayments r ON l.loan_id = r.loan_id
LEFT JOIN lending_club.loans_defaulters_delinq d ON c.member_id = d.member_id
LEFT JOIN lending_club.loans_defaulters_detail_rec_enq e ON c.member_id = e.member_id
""")

In [ ]:
spark.sql("select * from lending_club.customers_loan_v").show()

In [ ]:
spark.sql("""
create table lending_club.customers_loan_t as select
l.loan_id,
c.member_id,
c.emp_title,
c.emp_length,
c.home_ownership,
c.annual_income,
c.address_state,
c.address_zipcode,
c.address_country,
c.grade,
c.sub_grade,
c.verification_status,
c.total_high_credit_limit,
c.application_type,
c.join_annual_income,
c.verification_status_joint,
l.loan_amount,
l.funded_amount,
l.loan_term_years,
l.interest_rate,
l.monthly_installment,
l.issue_date,
l.loan_status,
l.loan_purpose,
r.total_principal_received,
r.total_interest_received,
r.total_late_fee_received,
r.last_payment_date,
r.next_payment_date,
d.delinq_2yrs,
d.delinq_amnt,
d.mths_since_last_delinq,
e.pub_rec,
e.pub_rec_bankruptcies,
e.inq_last_6mths

FROM lending_club.customers c
LEFT JOIN lending_club.loans l on c.member_id = l.member_id
LEFT JOIN lending_club.loans_repayments r ON l.loan_id = r.loan_id
LEFT JOIN lending_club.loans_defaulters_delinq d ON c.member_id = d.member_id
LEFT JOIN lending_club.loans_defaulters_detail_rec_enq e ON c.member_id = e.member_id
""")

In [ ]:
spark.sql("select * from lending_club.customers_loan_t").show()   

Remove duplicate member ids

In [5]:
spark.sql("""select member_id, count(*) as total 
from lending_club.customers
group by member_id order by total desc
""").show()

+--------------------+-----+
|           member_id|total|
+--------------------+-----+
|e4c167053d5418230...|    5|
|3f87585a20f702838...|    4|
|76b577467eda5bdbc...|    4|
|ad8e5d384dae17e06...|    4|
|c80b3e1938d2f7fad...|    3|
|bca141343d9405a9a...|    3|
|8cad841542bf57ce7...|    3|
|e7d8d16928817ec8f...|    3|
|f384e43664deed9a6...|    3|
|6f88761291d14b65e...|    3|
|5cd1a7e712c365a11...|    3|
|066ddaa64bee66dff...|    3|
|e2f7f066058696e76...|    3|
|092e68fe2b907d655...|    3|
|1a594f95f63362b33...|    3|
|22593a1870543b2db...|    3|
|63b021599856dc21b...|    3|
|ca5fd93b4f9adf941...|    3|
|f8a900a9b2359b5d6...|    3|
|52f6455c1b3b75d12...|    3|
+--------------------+-----+
only showing top 20 rows


In [ ]:
spark.sql("""select member_id, count(*) as total 
from lending_club.loans_defaulters_delinq
group by member_id order by total desc
""").show()

In [ ]:
spark.sql("""select member_id, count(*) as total 
from lending_club.loans_defaulters_detail_rec_enq
group by member_id order by total desc
""").show()

In [6]:
bad_data_customer_df = spark.sql("""select member_id from(select member_id, count(*)
as total from lending_club.customers
group by member_id having total > 1)""")

In [7]:
bad_data_customer_df.show()

+--------------------+
|           member_id|
+--------------------+
|cd1fdca829c443fa7...|
|cbede54df344cdb94...|
|f99c6e9cfe3a7a2d2...|
|2bae2e4dd6d5f2b21...|
|4231a55d0199c619a...|
|eebbd89aa7efc13e7...|
|d12c5766068d3b377...|
|d34249ef84980fb46...|
|3b94fbab00b7a0ec4...|
|e884f4108b3c6b1f4...|
|22593a1870543b2db...|
|a4327ad776a277161...|
|827a048c072fb44b0...|
|76b577467eda5bdbc...|
|7260b4e5a6d9fd42f...|
|bbe43331566910d55...|
|a596e2006964267d6...|
|df51e1f6c20da2046...|
|03f71ce106a7d33b3...|
|981a0200552d3e4c1...|
+--------------------+
only showing top 20 rows


In [8]:
bad_data_customer_df.count()

3157

In [9]:
bad_data_loans_defaulters_delinq_df = spark.sql("""select member_id from(select member_id, count(*)
as total from lending_club.loans_defaulters_delinq
group by member_id having total > 1)""")

In [10]:
bad_data_loans_defaulters_delinq_df.count()

939

In [11]:
bad_data_loans_defaulters_detail_rec_enq_df = spark.sql("""select member_id from(select member_id, count(*)
as total from lending_club.loans_defaulters_detail_rec_enq
group by member_id having total > 1)""")

bad_data_loans_defaulters_detail_rec_enq_df.count()

3189

In [ ]:
bad_data_customer_df.repartition(1).write \
.format("csv") \
.option("header", True) \
.mode("overwrite") \
.option("path", "../data/bad/bad_data_customers.csv") \
.save()

In [13]:
bad_data_loans_defaulters_delinq_df.repartition(1).write \
.format("csv") \
.option("header", True) \
.mode("overwrite") \
.option("path", "../data/bad/bad_data_loans_defaulters_delinq.csv") \
.save()

In [12]:
bad_customer_data_df = bad_data_customer_df.select("member_id") \
.union(bad_data_loans_defaulters_delinq_df.select("member_id")) \
.union(bad_data_loans_defaulters_detail_rec_enq_df.select("member_id"))

In [ ]:
bad_customer_data_final_df = bad_customer_data_df.distinct()


In [15]:
bad_customer_data_final_df.count()
bad_customer_data_final_df.show()

+--------------------+
|           member_id|
+--------------------+
|cd1fdca829c443fa7...|
|cbede54df344cdb94...|
|f99c6e9cfe3a7a2d2...|
|2bae2e4dd6d5f2b21...|
|4231a55d0199c619a...|
|eebbd89aa7efc13e7...|
|fc0e468bff11ac7c3...|
|7115ace193e13d8f3...|
|f284044b881f218c0...|
|01b2223757c3b62e7...|
|3b8ffe19f24749a73...|
|5130d0087970e032e...|
|a2af7506825a7dcff...|
|a53e2f488d2d76a30...|
|d4782ddad8591f44d...|
|675151e58a628e87b...|
|61be6498c93cadf89...|
|761b2f1e61999e14e...|
|d12c5766068d3b377...|
|d34249ef84980fb46...|
+--------------------+
only showing top 20 rows


In [16]:
bad_customer_data_final_df.repartition(1).write \
.format("csv") \
.option("header", True) \
.mode("overwrite") \
.option("path", "../data/bad/bad_customer_data_final.csv") \
.save()

In [17]:
bad_customer_data_final_df.createOrReplaceTempView("bad_data_customer")

In [18]:
customers_df = spark.sql("""select * from lending_club.customers
where member_id NOT IN (select member_id from bad_data_customer)
""")


In [19]:
customers_df.show()

+--------------------+--------------------+----------+--------------+-------------+-------------+---------------+---------------+-----+---------+-------------------+-----------------------+----------------+------------------+-------------------------+--------------------+
|           member_id|           emp_title|emp_length|home_ownership|annual_income|address_state|address_zipcode|address_country|grade|sub_grade|verification_status|total_high_credit_limit|application_type|join_annual_income|verification_status_joint|         ingest_date|
+--------------------+--------------------+----------+--------------+-------------+-------------+---------------+---------------+-----+---------+-------------------+-----------------------+----------------+------------------+-------------------------+--------------------+
|a5228d20be9d3871d...| Pharmacy Technician|         1|      MORTGAGE|      31000.0|           TX|          751xx|            USA|    G|       G1|    Source Verified|                

In [20]:
customers_df.write \
.format("parquet") \
.mode("overwrite") \
.option("path", "../data/cleaned_new/customers_parquet") \
.save()

In [21]:
temp = spark.read.parquet("../data/cleaned_new/customers_parquet")
temp.show()

+--------------------+--------------------+----------+--------------+-------------+-------------+---------------+---------------+-----+---------+-------------------+-----------------------+----------------+------------------+-------------------------+--------------------+
|           member_id|           emp_title|emp_length|home_ownership|annual_income|address_state|address_zipcode|address_country|grade|sub_grade|verification_status|total_high_credit_limit|application_type|join_annual_income|verification_status_joint|         ingest_date|
+--------------------+--------------------+----------+--------------+-------------+-------------+---------------+---------------+-----+---------+-------------------+-----------------------+----------------+------------------+-------------------------+--------------------+
|5241d9b258233d341...|   Clinical Director|         1|      MORTGAGE|      72000.0|           IN|          462xx|            USA|    A|       A5|       Not Verified|               1

In [22]:
loans_defaulters_delinq_df = spark.sql("""select * from lending_club.loans_defaulters_delinq
where member_id NOT IN (select member_id from bad_data_customer)
""")

In [23]:
loans_defaulters_delinq_df.show()

+--------------------+-----------+-----------+----------------------+
|           member_id|delinq_2yrs|delinq_amnt|mths_since_last_delinq|
+--------------------+-----------+-----------+----------------------+
|9cb79aa7323e81be1...|          2|        0.0|                    11|
|aac68850fdac09fd0...|          1|        0.0|                    21|
|c89986155a070db2e...|          1|        0.0|                     5|
|6e8d94bf446e97025...|          0|        0.0|                    36|
|42f73fd8a01f1c475...|          0|        0.0|                    46|
|1eef79a0e79b72c7a...|          1|        0.0|                    21|
|1dd1d1b51473d4993...|          0|        0.0|                    44|
|ec1953dba2cfb89ad...|          2|        0.0|                    13|
|8241a6bb3a9350fb8...|          0|        0.0|                    57|
|cdc94fa1c29a6a70a...|          0|        0.0|                    44|
|3712c9da85e54b7b1...|          1|        0.0|                    19|
|6ebc82410b3dc9dcb..

In [24]:
loans_defaulters_delinq_df.write \
.format("parquet") \
.mode("overwrite") \
.option("path", "../data/cleaned_new/loans_defaulters_delinq_parquet") \
.save()

In [25]:
temp = spark.read.parquet("../data/cleaned_new/loans_defaulters_delinq_parquet")
temp.show()

+--------------------+-----------+-----------+----------------------+
|           member_id|delinq_2yrs|delinq_amnt|mths_since_last_delinq|
+--------------------+-----------+-----------+----------------------+
|9cb79aa7323e81be1...|          2|        0.0|                    11|
|aac68850fdac09fd0...|          1|        0.0|                    21|
|c89986155a070db2e...|          1|        0.0|                     5|
|6e8d94bf446e97025...|          0|        0.0|                    36|
|42f73fd8a01f1c475...|          0|        0.0|                    46|
|1eef79a0e79b72c7a...|          1|        0.0|                    21|
|1dd1d1b51473d4993...|          0|        0.0|                    44|
|ec1953dba2cfb89ad...|          2|        0.0|                    13|
|8241a6bb3a9350fb8...|          0|        0.0|                    57|
|cdc94fa1c29a6a70a...|          0|        0.0|                    44|
|3712c9da85e54b7b1...|          1|        0.0|                    19|
|6ebc82410b3dc9dcb..

In [26]:
loans_defaulters_detail_rec_enq_df = spark.sql("""select * from lending_club.loans_defaulters_detail_rec_enq
where member_id NOT IN (select member_id from bad_data_customer)
""")

In [27]:
loans_defaulters_detail_rec_enq_df.write \
.format("parquet") \
.mode("overwrite") \
.option("path", "../data/cleaned_new/loans_defaulters_detail_rec_enq_parquet") \
.save()

In [28]:
temp = spark.read.parquet("../data/cleaned_new/loans_defaulters_detail_rec_enq_parquet")
temp.show()

+--------------------+-------+--------------------+--------------+
|           member_id|pub_rec|pub_rec_bankruptcies|inq_last_6mths|
+--------------------+-------+--------------------+--------------+
|9cb79aa7323e81be1...|      0|                   0|             0|
|0dd2bbc517e3c8f9e...|      1|                   1|             3|
|458458599d3df3bfc...|      1|                   1|             1|
|05ea141ec28b5c7f7...|      0|                   0|             0|
|aac68850fdac09fd0...|      0|                   0|             0|
|3a423e4589e89f429...|      0|                   0|             0|
|f1efcf7dfbfef21be...|      0|                   0|             1|
|c89986155a070db2e...|      0|                   0|             1|
|118dc629b6e134419...|      0|                   0|             0|
|a86fa4b7493708333...|      0|                   0|             0|
|6e8d94bf446e97025...|      0|                   0|             0|
|3de585156dc6b73f6...|      0|                   0|           

In [45]:
spark.sql("DROP TABLE IF EXISTS lending_club.loans_defaulters_detail_rec_enq_new")

DataFrame[]

In [39]:
spark.sql("""
create EXTERNAL TABLE lending_club.customers_new(
    member_id string, emp_title string, emp_length int, home_ownership string, 
    annual_income float, address_state string, address_zipcode string, 
    address_country string, grade string, sub_grade string, verification_status string,
    total_high_credit_limit float, application_type string, 
    join_annual_income float, verification_status_joint string, ingest_date timestamp)
    stored as parquet
    LOCATION '/Users/himanshuhegde/Desktop/Lending_club/data/cleaned_new/customers_parquet'
    
""")

DataFrame[]

In [40]:
spark.sql("select * from lending_club.customers_new").show()

+--------------------+--------------------+----------+--------------+-------------+-------------+---------------+---------------+-----+---------+-------------------+-----------------------+----------------+------------------+-------------------------+--------------------+
|           member_id|           emp_title|emp_length|home_ownership|annual_income|address_state|address_zipcode|address_country|grade|sub_grade|verification_status|total_high_credit_limit|application_type|join_annual_income|verification_status_joint|         ingest_date|
+--------------------+--------------------+----------+--------------+-------------+-------------+---------------+---------------+-----+---------+-------------------+-----------------------+----------------+------------------+-------------------------+--------------------+
|5241d9b258233d341...|   Clinical Director|         1|      MORTGAGE|      72000.0|           IN|          462xx|            USA|    A|       A5|       Not Verified|               1

In [43]:
spark.sql("""
create EXTERNAL TABLE lending_club.loans_defaulters_delinq_new(
    member_id string,delinq_2yrs integer, delinq_amnt float, mths_since_last_delinq integer)
    stored as parquet
    LOCATION '/Users/himanshuhegde/Desktop/Lending_club/data/cleaned_new/loans_defaulters_delinq_parquet'
""")

DataFrame[]

In [44]:
spark.sql("select * from lending_club.loans_defaulters_delinq_new").show()

+--------------------+-----------+-----------+----------------------+
|           member_id|delinq_2yrs|delinq_amnt|mths_since_last_delinq|
+--------------------+-----------+-----------+----------------------+
|9cb79aa7323e81be1...|          2|        0.0|                    11|
|aac68850fdac09fd0...|          1|        0.0|                    21|
|c89986155a070db2e...|          1|        0.0|                     5|
|6e8d94bf446e97025...|          0|        0.0|                    36|
|42f73fd8a01f1c475...|          0|        0.0|                    46|
|1eef79a0e79b72c7a...|          1|        0.0|                    21|
|1dd1d1b51473d4993...|          0|        0.0|                    44|
|ec1953dba2cfb89ad...|          2|        0.0|                    13|
|8241a6bb3a9350fb8...|          0|        0.0|                    57|
|cdc94fa1c29a6a70a...|          0|        0.0|                    44|
|3712c9da85e54b7b1...|          1|        0.0|                    19|
|6ebc82410b3dc9dcb..

In [46]:
spark.sql("""
create EXTERNAL TABLE lending_club.loans_defaulters_detail_rec_enq_new(
    member_id string, pub_rec integer, pub_rec_bankruptcies integer, inq_last_6mths integer)
    stored as parquet
    LOCATION '/Users/himanshuhegde/Desktop/Lending_club/data/cleaned_new/loans_defaulters_detail_rec_enq_parquet'
""")

DataFrame[]

In [47]:
spark.sql("select * from lending_club.loans_defaulters_detail_rec_enq_new").show()

+--------------------+-------+--------------------+--------------+
|           member_id|pub_rec|pub_rec_bankruptcies|inq_last_6mths|
+--------------------+-------+--------------------+--------------+
|9cb79aa7323e81be1...|      0|                   0|             0|
|0dd2bbc517e3c8f9e...|      1|                   1|             3|
|458458599d3df3bfc...|      1|                   1|             1|
|05ea141ec28b5c7f7...|      0|                   0|             0|
|aac68850fdac09fd0...|      0|                   0|             0|
|3a423e4589e89f429...|      0|                   0|             0|
|f1efcf7dfbfef21be...|      0|                   0|             1|
|c89986155a070db2e...|      0|                   0|             1|
|118dc629b6e134419...|      0|                   0|             0|
|a86fa4b7493708333...|      0|                   0|             0|
|6e8d94bf446e97025...|      0|                   0|             0|
|3de585156dc6b73f6...|      0|                   0|           

In [48]:
spark.sql("""select member_id, count(*) as total 
from lending_club.customers_new
group by member_id order by total desc""").show()

+--------------------+-----+
|           member_id|total|
+--------------------+-----+
|9bd211f8d8904dc92...|    1|
|906beba9eb9214071...|    1|
|fd2a69226e513b826...|    1|
|7ef1c352f8045bfcb...|    1|
|f399ad68d81e2ec20...|    1|
|1e354f59b6a33208a...|    1|
|017ce564dc0d6f975...|    1|
|a1f81d3435096cbf1...|    1|
|f73b6697408e3d625...|    1|
|41c7328b589c8b1bf...|    1|
|2ef59878a45a822c2...|    1|
|012779a4500cbb167...|    1|
|720650e8e7f58dcc6...|    1|
|23f0dad62fb95567c...|    1|
|57614ac711f005391...|    1|
|577c6cc337d82c5c3...|    1|
|5e761a67c6dcb64f8...|    1|
|ed5bfd4afec67ff05...|    1|
|ee2a75bb99e791188...|    1|
|c315fa535fe03136a...|    1|
+--------------------+-----+
only showing top 20 rows


Calculating loan scores